In [41]:
# INSTALL LIBRARIES
!pip install transformers torch seqeval

In [42]:
# IMPORT LIBRARIES
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

In [43]:
# CREATE CUSTOM DATASET
# Sample sentences
sentences = [
    ["Monis", "works", "at", "Google"],
    ["Ravi", "lives", "in", "New", "York"],
    ["Apple", "is", "a", "tech", "company"]
]

# POS Tags
pos_tags = [
    ["NNP", "VBZ", "IN", "NNP"],
    ["NNP", "VBZ", "IN", "NNP", "NNP"],
    ["NNP", "VBZ", "DT", "NN", "NN"]
]

# Chunk Tags
chunk_tags = [
    ["B-NP", "B-VP", "B-PP", "B-NP"],
    ["B-NP", "B-VP", "B-PP", "B-NP", "I-NP"],
    ["B-NP", "B-VP", "B-NP", "I-NP", "I-NP"]
]

In [44]:
# CREATE LABEL MAP
label_list = list(set([tag for seq in chunk_tags for tag in seq]))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

In [45]:
# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [46]:
# TOKENIZE + ALIGN LABELS
def tokenize_and_align(sentences, labels):
    input_ids = []
    attention_masks = []
    label_ids = []

    for sent, lab in zip(sentences, labels):
        encoding = tokenizer(sent, is_split_into_words=True, padding='max_length', truncation=True, max_length=10)

        word_ids = encoding.word_ids()
        aligned_labels = []

        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(label2id[lab[word_idx]])
            else:
                aligned_labels.append(-100)

            previous_word_idx = word_idx

        input_ids.append(encoding["input_ids"])
        attention_masks.append(encoding["attention_mask"])
        label_ids.append(aligned_labels)

    return input_ids, attention_masks, label_ids

In [47]:
# PREPARE DATA
input_ids, attention_masks, labels = tokenize_and_align(sentences, chunk_tags)

In [48]:
# CREATE DATASET CLASS
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_masks[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

train_dataset = CustomDataset(input_ids, attention_masks, labels)

In [49]:
# MODEL SETUP
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [50]:
# TRAINING CONFIG
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [51]:
# METRICS
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_preds = []
        temp_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                temp_preds.append(id2label[p_val])
                temp_labels.append(id2label[l_val])

        true_preds.append(temp_preds)
        true_labels.append(temp_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }

In [52]:
# TRAIN MODEL
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10, training_loss=1.2331005096435548, metrics={'train_runtime': 15.2078, 'train_samples_per_second': 0.986, 'train_steps_per_second': 0.658, 'total_flos': 38278659600.0, 'train_loss': 1.2331005096435548, 'epoch': 5.0})

In [53]:
# INFERENCE
sentence = ["John", "works", "at", "Google"]

inputs = tokenizer(sentence, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs).logits
predictions = torch.argmax(outputs, dim=2)

predicted_labels = [id2label[p.item()] for p in predictions[0]]

print(list(zip(sentence, predicted_labels)))

[('John', 'B-NP'), ('works', 'B-PP'), ('at', 'B-NP'), ('Google', 'B-NP')]


**REPORT:**

POS Tagging:
Identifies grammatical roles (noun, verb)

Chunking:
Identifies phrases (NP, VP)

Difference:
POS → word-level
Chunking → phrase-level

Challenges:
- Label alignment
- Small dataset
- Handling subwords

Conclusion:
Transformers perform well for token classification tasks.